In [0]:
# Test basic S3 access
try:
    dbutils.fs.ls("s3://adtech-optimizer-data/")
    print("✅ S3 access successful!")
except Exception as e:
    print(f"❌ Error: {e}")

✅ S3 access successful!


In [0]:
# Databricks notebook source
# ============================================================
# EXPORT GOLD TABLE TO AWS S3
# ============================================================
# Purpose: Export Gold table to S3 as Parquet with validation

from pyspark.sql.functions import *
from datetime import datetime
import yaml
import os

# ============================================================
# 1. LOAD CONFIGURATION
# ============================================================

def load_yaml_config():
    """Load pipeline configuration from YAML file"""
    try:
        try:
            with open("pipeline_manifest.yaml", "r") as f:
                config = yaml.safe_load(f)
                print("Loaded config from local path")
                return config
        except:
            pass

        try:
            config_path = "/Volumes/adtech_catalog/bronze/landing_zone/pipeline_manifest.yaml"
            try:
                dbutils.fs.ls(config_path)
                config_content = dbutils.fs.head(config_path)
                config = yaml.safe_load(config_content)
                print(f"Loaded config from: {config_path}")
                return config
            except:
                print("Config file not found in DBFS")
                return None
        except:
            return None

    except Exception as e:
        print(f"Could not load config: {e}")
        return None

config = load_yaml_config()

ENVIRONMENT = config.get('environment', 'development') if config else 'development'
VERSION = datetime.now().strftime("%Y%m%d_%H%M%S")
GIT_COMMIT = os.environ.get('GIT_COMMIT', 'local')

print("="*70)
print("CONFIGURATION SUMMARY")
print("="*70)
print(f"Environment: {ENVIRONMENT}")
print(f"Version: {VERSION}")
print(f"Git Commit: {GIT_COMMIT}")
print("="*70)

# ============================================================
# 2. LOAD GOLD DATA WITH IDEMPOTENCY CHECK
# ============================================================

print("="*70)
print("EXPORTING GOLD DATA TO S3")
print("="*70)

def check_table_exists(table_name):
    try:
        spark.sql(f"DESCRIBE {table_name}")
        return True
    except:
        return False

GOLD_TABLE = "adtech_catalog.gold.fact_ad_performance"

if not check_table_exists(GOLD_TABLE):
    print(f"ERROR: Table {GOLD_TABLE} does not exist.")
    print("Please run 04_FEATURE_ENGINEERING.py first.")
    dbutils.notebook.exit("Gold table not found: fact_ad_performance")

df_gold = spark.table(GOLD_TABLE)

total_rows = df_gold.count()
total_cols = len(df_gold.columns)

print(f"""
GOLD DATA SUMMARY
======================================================================
Table: {GOLD_TABLE}
Total Rows: {total_rows:,}
Total Columns: {total_cols}
======================================================================
""")

# ============================================================
# 3. ADD PARTITION COLUMNS
# ============================================================

print("")
print("Adding partition columns...")

current_ts = datetime.now()

df_export = df_gold.withColumn(
    "year", year("processing_date")
).withColumn(
    "month", month("processing_date")
).withColumn(
    "day", dayofmonth("processing_date")
).withColumn(
    "export_timestamp", lit(current_ts.isoformat())
).withColumn(
    "export_version", lit(VERSION)
).withColumn(
    "environment", lit(ENVIRONMENT)
)

print("Partition columns: year, month, day")
print(f"Export Version: {VERSION}")
print(f"Environment: {ENVIRONMENT}")

# ============================================================
# 4. CONFIGURE S3 PATHS
# ============================================================

print("")
print("Configuring S3 paths...")

# New bucket name - more professional and descriptive
BUCKET_NAME = "adtech-optimizer-data"
S3_PATH = f"s3://{BUCKET_NAME}/gold/fact_ad_performance/"

print(f"S3 Path: {S3_PATH}")
print(f"Bucket: {BUCKET_NAME}")

# ============================================================
# 5. CHECK IF DATA ALREADY EXISTS
# ============================================================

print("")
print("Checking if data already exists...")

try:
    existing_files = dbutils.fs.ls(S3_PATH)
    print(f"Existing data found at: {S3_PATH}")
    print("It will be OVERWRITTEN")
except Exception as e:
    print("No existing data found. Creating new.")

# ============================================================
# 6. WRITE TO S3
# ============================================================

print("")
print("Writing to S3...")

try:
    df_export.write \
        .mode("overwrite") \
        .format("parquet") \
        .option("compression", "snappy") \
        .partitionBy("year", "month", "day") \
        .save(S3_PATH)

    print("Data exported successfully!")
    print(f"   Path: {S3_PATH}")
    print(f"   Rows: {total_rows:,}")

except Exception as e:
    print(f"Export failed: {e}")
    print("")
    print("Troubleshooting:")
    print("1. Check IAM role permissions")
    print("2. Check bucket exists: adtech-optimizer-data")
    print("3. Check External Location configuration")
    raise

# ============================================================
# 7. VERIFY EXPORT
# ============================================================

print("")
print("Verifying export...")

try:
    df_verify = spark.read.parquet(S3_PATH)
    verify_count = df_verify.count()
    verify_cols = len(df_verify.columns)

    print("Verification successful!")
    print(f"   Rows in S3: {verify_count:,}")
    print(f"   Columns in S3: {verify_cols}")

    if verify_count == total_rows:
        print("   Row count matches Gold table!")
    else:
        print(f"   Row count mismatch. Expected: {total_rows}, Found: {verify_count}")

except Exception as e:
    print(f"Verification failed: {e}")

# ============================================================
# 8. LIST FILES IN S3
# ============================================================

print("")
print("Files in S3:")

try:
    files = dbutils.fs.ls(S3_PATH)
    print(f"   Total items: {len(files)}")

    parquet_files = [f for f in files if f.name.endswith('.parquet')]
    print(f"   Parquet files: {len(parquet_files)}")

    if parquet_files:
        print("")
        print("   Sample files:")
        for f in parquet_files[:5]:
            size_mb = f.size / (1024 * 1024)
            print(f"   - {f.name} ({size_mb:.2f} MB)")

    if len(parquet_files) > 5:
        print(f"   ... and {len(parquet_files) - 5} more files")

except Exception as e:
    print(f"Could not list files: {e}")

# ============================================================
# 9. SAMPLE DATA FROM S3
# ============================================================

print("")
print("Sample data from S3:")

try:
    df_sample = spark.read.parquet(S3_PATH).limit(5)
    display(df_sample)
except Exception as e:
    print(f"Could not display sample: {e}")

# ============================================================
# 10. SAVE EXPORT METADATA
# ============================================================

print("")
print("Saving export metadata...")

try:
    spark.sql("CREATE SCHEMA IF NOT EXISTS adtech_catalog.monitoring")

    export_metadata = spark.createDataFrame([(
        current_ts.isoformat(),
        S3_PATH,
        total_rows,
        total_cols,
        "fact_ad_performance",
        VERSION,
        ENVIRONMENT,
        GIT_COMMIT,
        "SUCCESS"
    )], [
        "export_timestamp",
        "s3_path",
        "row_count",
        "column_count",
        "table_name",
        "export_version",
        "environment",
        "git_commit",
        "status"
    ])

    export_metadata.write \
        .mode("append") \
        .format("delta") \
        .saveAsTable("adtech_catalog.monitoring.s3_export_log")

    print("Export metadata saved to: adtech_catalog.monitoring.s3_export_log")
    print(f"   Export Version: {VERSION}")
    print(f"   Environment: {ENVIRONMENT}")

except Exception as e:
    print(f"Could not save export metadata: {e}")

# ============================================================
# 11. SAVE VERSION HISTORY
# ============================================================

print("")
print("Saving version history...")

try:
    version_info = spark.createDataFrame([(
        VERSION,
        ENVIRONMENT,
        GIT_COMMIT,
        datetime.now().isoformat(),
        "Export Gold to S3",
        "SUCCESS"
    )], [
        "version_id",
        "environment",
        "git_commit",
        "deployed_at",
        "description",
        "status"
    ])

    version_info.write \
        .mode("append") \
        .format("delta") \
        .saveAsTable("adtech_catalog.monitoring.version_history")

    print("Version history updated: adtech_catalog.monitoring.version_history")
    print(f"   Version: {VERSION}")

except Exception as e:
    print(f"Could not save version history: {e}")

# ============================================================
# 12. SUMMARY
# ============================================================

print("")
print("="*70)
print("EXPORT COMPLETE")
print("="*70)

print(f"""
EXPORT SUMMARY
======================================================================
Version: {VERSION}
Environment: {ENVIRONMENT}

S3 Path: {S3_PATH}
Rows: {total_rows:,}
Columns: {total_cols}
Format: Parquet (Snappy compression)
Partitions: year/month/day
Export Version: {VERSION}

S3 Bucket: {BUCKET_NAME}

Folder Structure:
   s3://{BUCKET_NAME}/
   └── gold/
       └── fact_ad_performance/
           ├── year=2026/
           │   └── month=07/
           │       └── day=31/
           │           └── part-*.snappy.parquet
           └── _SUCCESS

Monitoring:
   - Export Log: adtech_catalog.monitoring.s3_export_log
   - Version History: adtech_catalog.monitoring.version_history

Next Steps:
   1. Verify data in AWS S3 Console
   2. Run: ML Training (01_CTR_PREDICTION.py)
   3. Build: Streamlit Dashboard
======================================================================
""")

Loaded config from: /Volumes/adtech_catalog/bronze/landing_zone/pipeline_manifest.yaml
CONFIGURATION SUMMARY
Environment: development
Version: 20260802_110305
Git Commit: local
EXPORTING GOLD DATA TO S3

GOLD DATA SUMMARY
Table: adtech_catalog.gold.fact_ad_performance
Total Rows: 9,999
Total Columns: 46


Adding partition columns...
Partition columns: year, month, day
Export Version: 20260802_110305
Environment: development

Configuring S3 paths...
S3 Path: s3://adtech-optimizer-data/gold/fact_ad_performance/
Bucket: adtech-optimizer-data

Checking if data already exists...
Existing data found at: s3://adtech-optimizer-data/gold/fact_ad_performance/
It will be OVERWRITTEN

Writing to S3...
Data exported successfully!
   Path: s3://adtech-optimizer-data/gold/fact_ad_performance/
   Rows: 9,999

Verifying export...
Verification successful!
   Rows in S3: 9,999
   Columns in S3: 51
   Row count matches Gold table!

Files in S3:
   Total items: 3
   Parquet files: 0

Sample data from S3:


Ad_Reference_ID,ad_category,ad_device,ad_location,ad_type,ad_type_catalog,cost_per_click,ad_video_length,total_clicks,total_impressions,ctr,avg_watch_duration,total_ad_spend,total_revenue,roas,total_conversions,conversion_rate,overall_conversion_rate,avg_user_age,unique_users,avg_watch_ratio,avg_ded_score,category_age_affinity,platforms_used,devices_used,platform_avg_roas,platform_total_spend,platform_total_revenue,active_time_slots,best_day,avg_hour,ingestion_date,ingestion_timestamp,engagement_efficiency,profit_margin,cost_per_conversion,high_performance,cost_efficiency_score,engagement_score,audience_alignment_score,ad_age_days,ad_lifecycle_stage,location_type,season,processing_date,export_timestamp,export_version,environment,year,month,day
AD_117417,Electronics,Tablet,Karnataka,Text,Image,1.88,0.0,2,215,0.009302325581395349,0.015813953488372095,3.76,0.0,0.0,0,0.0,0.0,42.07906976744186,215,0.0,0.09999999999999992,0.016279391278817973,"List(google, instagram, Unknown, facebook)","List(Tablet, Mobile, Desktop, Unknown)",0.004139926091193895,10036.52,2810.527602477398,"List(Afternoon, Evening_Prime, Late_Night, Morning)",5,11.962790697674418,2026-08-02,2026-08-02T10:11:18.245Z,0.0,-1.0,0E-9,0,0.0,0.0037209302325581397,0.001627939127881796,0,New,Urban,Winter,2026-08-02,2026-08-02T11:03:05.998933,20260802_110305,development,2026,8,2
AD_521967,Electronics,All-Devices,Delhi,Text,Video,3.16,30.0,4,196,0.02040816326530612,5.902551020408162,12.64,28.942578674033378,2.2897609710469444,1,0.25,0.00510204081632653,41.94387755102041,195,0.19675170068027212,0.09999999999999994,0.016370775937616752,"List(instagram, facebook, Unknown, google)","List(Unknown, Mobile, Tablet, Desktop)",0.004139926091193895,10036.52,2810.527602477398,"List(Late_Night, Morning, Afternoon, Evening_Prime)",3,10.392857142857142,2026-08-02,2026-08-02T10:11:18.245Z,0.004015340830209635,1.2897609710469444,12.640000000,1,0.001614569878584345,0.1421887755102041,0.0016370775937616742,0,New,Urban,Spring,2026-08-02,2026-08-02T11:03:05.998933,20260802_110305,development,2026,8,2
AD_578928,Electronics,Desktop,Karnataka,Video,Image,3.21,0.0,4,195,0.020512820512820513,0.03641025641025641,12.84,0.0,0.0,0,0.0,0.0,40.15384615384615,195,0.0,0.09999999999999995,0.01682097602891295,"List(facebook, google, instagram, Unknown)","List(Mobile, Tablet, Desktop, Unknown)",0.004139926091193895,10036.52,2810.527602477398,"List(Late_Night, Afternoon, Evening_Prime, Morning)",6,11.189743589743589,2026-08-02,2026-08-02T10:11:18.245Z,0.0,-1.0,0E-9,0,0.0,0.008205128205128205,0.0016820976028912942,0,New,Urban,Winter,2026-08-02,2026-08-02T11:03:05.998933,20260802_110305,development,2026,8,2
AD_512417,Electronics,Desktop,Delhi,Image,Video,3.44,30.0,2,187,0.0106951871657754,5.638502673796791,6.88,0.0,0.0,0,0.0,0.0,42.05882352941177,186,0.18795008912655975,0.09999999999999992,0.016483084258649586,"List(instagram, facebook, Unknown, google)","List(Mobile, Tablet, Unknown, Desktop)",0.004139926091193895,10036.52,2810.527602477398,"List(Late_Night, Evening_Prime, Afternoon, Morning)",7,11.101604278074866,2026-08-02,2026-08-02T10:11:18.245Z,0.0020101613810327244,-1.0,0E-9,0,0.0,0.06066310160427808,0.0016483084258649573,0,New,Urban,Spring,2026-08-02,2026-08-02T11:03:05.998933,20260802_110305,development,2026,8,2
AD_478798,Electronics,Tablet,Karnataka,Image,Carousel,1.97,0.0,2,236,0.00847457627118644,0.027966101694915254,3.94,0.0,0.0,0,0.0,0.0,42.847457627118644,236,0.0,0.09999999999999992,0.016117721830710197,"List(instagram, Unknown, google, facebook)","List(Unknown, Tablet, Mobile, Desktop)",0.004139926091193895,10036.52,2810.527602477398,"List(Afternoon, Evening_Prime, Late_Night, Morning)",5,11.572033898305085,2026-08-02,2026-08-02T10:11:18.245Z,0.0,-1.0,0E-9,0,0.0,0.0033898305084745766,0.0016117721830710184,0,New,Urban,Winter,2026-08-02,2026-08-02T11:03:05.998933,20260802_110305,development,2026,8,2



Saving export metadata...
Export metadata saved to: adtech_catalog.monitoring.s3_export_log
   Export Version: 20260802_110305
   Environment: development

Saving version history...
Version history updated: adtech_catalog.monitoring.version_history
   Version: 20260802_110305

EXPORT COMPLETE

EXPORT SUMMARY
Version: 20260802_110305
Environment: development

S3 Path: s3://adtech-optimizer-data/gold/fact_ad_performance/
Rows: 9,999
Columns: 46
Format: Parquet (Snappy compression)
Partitions: year/month/day
Export Version: 20260802_110305

S3 Bucket: adtech-optimizer-data

Folder Structure:
   s3://adtech-optimizer-data/
   └── gold/
       └── fact_ad_performance/
           ├── year=2026/
           │   └── month=07/
           │       └── day=31/
           │           └── part-*.snappy.parquet
           └── _SUCCESS

Monitoring:
   - Export Log: adtech_catalog.monitoring.s3_export_log
   - Version History: adtech_catalog.monitoring.version_history

Next Steps:
   1. Verify data in 